In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

In [ ]:
import glob
import pandas as pd

files = glob.glob("../data/*.csv")

df = pd.concat((pd.read_csv(file) for file in files), ignore_index=True)

df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.dtypes

In [ ]:
df["start_time"] = pd.to_datetime(df["start_time"])
df["end_time"] = pd.to_datetime(df["end_time"])

In [ ]:
df.info()

In [ ]:
#NEW FEATURES
df["year"] = df["start_time"].dt.year
df["month"] = df["start_time"].dt.month
df["day"] = df["start_time"].dt.day
df["hour"] = df["start_time"].dt.hour
df["weekday"] = df["start_time"].dt.day_name()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
#USER TYPE DISTRIBUTION
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(data=df, x="user_type")
plt.xticks(rotation=45)
plt.show()

In [ ]:
#GENDRE DISTRIBUTION
sns.countplot(data=df, x="member_gender")
plt.show()

In [ ]:
#RIDE DISTRIBUTION DURATION
plt.figure(figsize=(10,5))
sns.histplot(df["duration_sec"], bins=50)
plt.show()

In [ ]:
#MONTHLY RIDES
sns.countplot(data=df, x="month")
plt.show()

In [ ]:
#HOURLY RIDES
plt.figure(figsize=(12,5))
sns.countplot(data=df, x="hour")
plt.show()


In [ ]:
#WEEKDAY RIDES
plt.figure(figsize=(10,5))
sns.countplot(data=df, x="weekday")
plt.xticks(rotation=45)
plt.show()

In [ ]:
#CORRELATION
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.show()

In [ ]:
df.to_csv("../output/cleaned_bike_data.csv", index=False)

In [ ]:
df["duration_min"] = df["duration_sec"] / 60

In [ ]:
df[["duration_sec", "duration_min"]].head()

In [ ]:
#RIDER AGE
df["age"] = 2018 - df["member_birth_year"]

In [ ]:
df["age"].describe()

In [ ]:
#REMOVE INVALID AGES
df = df[(df["age"] > 10) & (df["age"] < 90)]

In [ ]:
df["age"].describe()

In [ ]:
#RIDE DURATION PLOT
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,5))
sns.histplot(df["duration_min"], bins=50)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df["age"], bins=30)
plt.show()

In [ ]:
#AVERAGE RIDE DURATION BY USER TYPE
df.groupby("user_type")["duration_min"].mean()

In [ ]:
#AVERAGE DURATION BY GENDER
df.groupby("member_gender")["duration_min"].mean()

In [ ]:
df.to_csv("../output/final_bike_data.csv", index=False)

In [ ]:
#HERE WE are STARTING MACHINE LEARNING
#REQUIRED LIBRARIES
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
#CATEGORICAL COLUMNS ENCODING
le = LabelEncoder()

df["user_type"] = le.fit_transform(df["user_type"])
df["member_gender"] = le.fit_transform(df["member_gender"])

In [ ]:
# FEATURES AND TARGET
#HUM USER_TYPE KO PREDICT KARENGE
X = df[["duration_min", "age", "hour", "month"]]
y = df["user_type"]

In [ ]:
df = df.sample(n=100000, random_state=42)

In [ ]:
#TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
#MODEL TRAIN 
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

In [ ]:
#PREDICTION
y_pred = model.predict(X_test)

In [ ]:
#ACCURACY
accuracy_score(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
#CONFUSION MATRIX
cm = confusion_matrix(y_test, y_pred)

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# FEATURE IMPORTANCE
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance.sort_values(by="Importance", ascending=False)

In [ ]:
import joblib

joblib.dump(model, "../output/random_forest_model.pkl")

In [ ]:
import pandas as pd

df = pd.read_csv("../output/final_bike_data.csv")

# 50,000 rows का sample
df_small = df.sample(n=50000, random_state=42)

df_small.to_csv("../output/final_bike_data_small.csv", index=False)

print("Done")